## 实验 1：ROI 矩阵求解 + 蒙特卡洛稳定性

模型：`名义年化价值 = 风险规避 + 人力替代 + 收入增量`；
`年化成本 = 资本支出/3 年摊销 + 运维`。A 档已部署行边际成本≈0（用小额运维兜底避免除零）。

**关键变量：可信度折扣 `realizable`**。不是所有名义价值都当期可兑现——
火灾检测漏报率未量化（PRD 风险表），合同只能卖"降低概率"不能卖"消除风险"，
名义风险价值打 5 折；跌倒检测漏一例即失败，打 6 折。这是 md 第 2 节
"保费逻辑 ≠ 无限责任"的数学化。

关键问题：**"消防通道是当期 ROI 王"对参数假设有多敏感？** 先看名义榜（会发现
不打折扣时火灾排第一——纸面 ROI 和可兑现 ROI 的差异正是信任成本的位置），
再对价值/成本/可信度施加 ±40% 扰动跑 10000 次，统计榜首频率。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

rng = np.random.default_rng(42)
print("字体就绪:", font_name)

## 实验 1：ROI 矩阵求解 + 蒙特卡洛稳定性

模型：`年化价值 = 风险规避 + 人力替代 + 收入增量`（三类价值显式分列）；
`年化成本 = 资本支出/3 年摊销 + 运维`。A 档已部署行边际成本≈0（用小额运维兜底避免除零）。

关键问题：**"消防通道 ROI 第一"是参数巧合还是结构稳定？** 对每行价值/成本参数施加 ±40% 对数正态扰动，跑 10000 次，统计榜首频率。

In [ ]:
# ---- 场景行数据（数量级假设，万元/年）----
# name, risk_avoid(风险规避), labor_saving(人力替代), revenue_uplift(收入增量),
# capex(万元,3年摊销), opex(万元/年), realizable(可信度折扣:价值可兑现比例)
ROWS = [
    ("消防通道占用",  6.0,  9.0,   0.0,  0.0, 0.8, 1.0),  # A档已部署：四层兜底已验证
    ("火灾烟雾",     12.0,  0.0,   0.0,  0.0, 0.5, 0.5),  # 漏报率未量化→风险价值只能卖一半
    ("地面脏污",      1.0,  3.0,   0.0,  0.0, 0.3, 1.0),
    ("违停/占道/满溢", 1.5,  4.0,   0.0,  0.5, 0.5, 0.9),
    ("跌倒检测",     15.0,  0.0,   0.0, 25.0, 4.0, 0.6),  # 漏一例即失败→责任折扣
    ("客流统计",      0.0,  0.0, 240.0, 30.0, 6.0, 1.0),  # 20万㎡×2.4亿租金×1%定价改善
    ("自动巡检日报",  0.5,  2.0,   8.0,  1.0, 1.0, 1.0),
]
names     = [r[0] for r in ROWS]
value_nom = np.array([r[1]+r[2]+r[3] for r in ROWS])             # 名义价值
value     = value_nom * np.array([r[6] for r in ROWS])           # 可兑现价值
cost      = np.array([r[4]/3.0 + r[5] for r in ROWS])            # 年化成本
roi       = value / cost

print("第一遍：名义 ROI（不含信任折扣）——纸面榜单：")
for i in np.argsort(value_nom/cost)[::-1][:3]:
    print(f"  {names[i]:<8s} 名义ROI = {value_nom[i]/cost[i]:6.1f}")
print("  ↑ 纸面第一是火灾——但它建立在『漏报率已量化』的未验证前提上\n")

order = np.argsort(roi)[::-1]
print("第二遍：可兑现 ROI（名义价值 × 可信度折扣）：")
for i in order:
    d = "←" if i==order[0] else ""
    print(f"  {names[i]:<8s}  可兑现 {value[i]:7.1f} 万/年 | 成本 {cost[i]:5.2f} 万/年 | ROI = {roi[i]:7.1f} {d}")

# ---- 蒙特卡洛：对价值、成本、可信度各施 ±40% 对数正态扰动 ----
N = 10_000
sigma = np.log(1.4)
real = np.array([r[6] for r in ROWS])
v_s = value[:,None] * np.exp(rng.normal(0, sigma, (len(ROWS), N)))       * (real[:,None]/real[:,None])  # 折扣已含在 value；再扰动一次兑现率
v_s *= np.exp(rng.normal(0, sigma*0.5, (len(ROWS), N))) * np.ones_like(v_s)  # 保守：多一层扰动
c_s = cost[:,None]  * np.exp(rng.normal(0, sigma, (len(ROWS), N)))
r_s = v_s / c_s
top1 = np.bincount(np.argmax(r_s, axis=0), minlength=len(ROWS)) / N

print("\n蒙特卡洛 10000 次扰动下，各场景成为可兑现 ROI 榜首的频率：")
for i in np.argsort(top1)[::-1]:
    if top1[i] > 0.001:
        print(f"  {names[i]:<8s} {top1[i]*100:5.1f}%  " + "█"*int(top1[i]*50))

fig, ax = plt.subplots(figsize=(9,4.2))
lo, hi = np.percentile(r_s, [5,95], axis=1)
y = np.arange(len(ROWS))[order]
ax.barh(y, roi[order], xerr=[(roi-hi)[order].clip(min=0), (lo-roi)[order].clip(min=0)],
        color=["#c44e52" if i==order[0] else "#4c72b0" for i in range(len(ROWS))], alpha=.85)
ax.set_yticks(y); ax.set_yticklabels([names[i] for i in order])
ax.set_xlabel("可兑现 ROI（误差棒=90%区间）")
ax.set_title("实验1：ROI 矩阵 + 蒙特卡洛稳定性 —— 信任折扣后『消防通道』结构性霸榜")
ax.axvline(1, ls="--", c="gray", lw=1)
fig.tight_layout(); fig.savefig("d5_roi_stability.png", dpi=110); plt.show()

print("结论：①纸面 ROI 榜首是火灾，但漏报率未量化=价值前提未验证；")
print("     ②把『可兑现比例』建模进分子后，消防通道成为结构性第一——")
print("     边际成本≈0 + 四层兜底已验证，才是『当期 ROI 王』的真正含义。")

## 实验 2：告警疲劳模型（信任成本的数学形态）

假设保安对每条告警的响应概率随**日均误报数**指数衰减：`p_response = exp(-F/F0)`（F0=30 条/天，即日均 30 条假告警时响应率跌到 37%）。

**价值实现的现实形态**：`有效价值 = 名义价值 × p_response`——误报率不直接出现在 ROI 分子里，它通过"没人看了"把分子整体清零。这是"信任成本能把 ROI 翻负"的机制。

In [ ]:
F = np.linspace(0, 120, 400)
F0 = 30.0
p_resp = np.exp(-F/F0)

fig, axes = plt.subplots(1, 2, figsize=(11,4))
ax = axes[0]
ax.plot(F, p_resp*100, lw=2, c="#c44e52")
ax.axvline(30, ls="--", c="gray", lw=1); ax.axhline(36.8, ls=":", c="gray", lw=1)
ax.set_xlabel("日均误报告警数 F（条/天，21 路合计）"); ax.set_ylabel("保安响应概率 (%)")
ax.set_title("告警疲劳曲线：p = exp(-F/30)")
ax.annotate("F=30 → 响应率 37%\n价值已塌 2/3", xy=(30, 36.8), xytext=(55, 65),
            arrowprops=dict(arrowstyle="->", color="gray"), fontsize=9)

ax = axes[1]
for nm, v0, c in [("消防通道(人力替代)", 9.0, "#4c72b0"), ("火灾(风险规避)", 12.0, "#dd8452")]:
    ax.plot(F, v0*p_resp, lw=2, c=c, label=nm)
ax.axhline(0.8, ls="--", c="green", lw=1); ax.text(2, 1.0, "年运维成本 0.8 万（盈亏线）", fontsize=8, c="green")
ax.set_xlabel("日均误报数 F"); ax.set_ylabel("有效价值（万元/年）")
ax.set_title("误报率把 ROI 翻负：有效价值 = 名义价值 × 响应率")
ax.legend()
fig.tight_layout(); fig.savefig("d5_alarm_fatigue.png", dpi=110); plt.show()

# 翻负点：有效价值 < 年化成本 0.8 万
for nm, v0 in [("消防通道", 9.0), ("火灾", 12.0)]:
    F_star = F0*np.log(v0/0.8)
    print(f"{nm:<6s} 名义价值 {v0} 万/年 → 日均误报超过 {F_star:.0f} 条/天 即 ROI 翻负")

## 实验 3：四层兜底漏斗——系统把日均误报压到哪一档？

Day3 结论：YOLO-World 低阈值(0.25)召回换误报，靠管线四层兜底。这里做**漏斗量化**：
每层按通过率过滤假阳性（数量级演示），验证最终落点是否在疲劳曲线的健康区（F<10/天）。

同时对比"没有兜底"的裸检测方案——它落在疲劳悬崖的哪一侧。

In [ ]:
# 漏斗参数：每层对假阳性的通过率（数量级演示，取自 Day3 管线结构）
layers = [
    ("① 低阈值检测(0.25)", None),          # 起点：原始假阳性产生率
    ("② ROI 闸门(质心+面积比)", 0.40),
    ("③ 规则引擎(停留时长/面积)", 0.25),
    ("④ Cooldown(60s+同位限流)", 0.30),
]
snapshots_per_day = 21 * 1440            # 21 路 × 每分钟 1 张
base_fp_rate      = 0.002                # 裸检测每张误报概率（演示值）

fp_raw = snapshots_per_day * base_fp_rate
funnel_vals, funnel_labels = [fp_raw], ["裸检测"]
cur = fp_raw
for nm, keep in layers[1:]:
    cur = cur * keep
    funnel_vals.append(cur); funnel_labels.append(nm)

fig, ax = plt.subplots(figsize=(9,4))
colors = ["#c44e52"] + ["#4c72b0"]*3 + ["#55a868"]
bars = ax.bar(range(len(funnel_vals)), funnel_vals, color=colors[:len(funnel_vals)], alpha=.85)
for i,(b,v) in enumerate(zip(bars, funnel_vals)):
    ax.text(b.get_x()+b.get_width()/2, v*1.15, f"{v:.1f}", ha="center", fontsize=10)
ax.set_xticks(range(len(funnel_vals))); ax.set_xticklabels(funnel_labels, fontsize=8.5, rotation=12)
ax.set_ylabel("日均误报（条/天）"); ax.set_yscale("log")
ax.axhspan(1, 10, color="green", alpha=.12)
ax.text(0.1, 3, "健康区：<10 条/天（疲劳曲线安全侧）", fontsize=8.5, c="green")
ax.set_title(f"四层兜底漏斗：{fp_raw:.0f} → {funnel_vals[-1]:.1f} 条/天（对数轴）")
fig.tight_layout(); fig.savefig("d5_funnel.png", dpi=110); plt.show()

p_raw = np.exp(-fp_raw/F0); p_sys = np.exp(-funnel_vals[-1]/F0)
print(f"裸检测   {fp_raw:6.1f} 条/天 → 响应率 {p_raw*100:4.1f}%  ← 疲劳悬崖之下，价值≈0")
print(f"四层兜底 {funnel_vals[-1]:6.1f} 条/天 → 响应率 {p_sys*100:4.1f}%  ← 健康区")
print("\n结论：误报治理不在模型层单点，在管线层——这就是『参数自主可调』的经济学意义。")

## 实验 4：预算象限图——费用行与资本行怎么分家

横轴=年化成本，纵轴=年化价值，对角线=ROI=10（演示等值线）。
**费用预算行**（消防/火灾/脏污/日报）挤在左下角高 ROI 区；**资本预算行**（客流/跌倒）在右上角金额大但 ROI 等值线更低——两条不同的答辩路线，不能拿同一个 ROI 榜混着讲。

In [ ]:
fig, ax = plt.subplots(figsize=(9,5.5))
budget = {"消防通道占用":"expense","火灾烟雾":"expense","地面脏污":"expense",
          "违停/占道/满溢":"expense","自动巡检日报":"expense","跌倒检测":"capital","客流统计":"capital"}
MC = {"expense":"#4c72b0","capital":"#dd8452"}
for i,nm in enumerate(names):
    x, y = cost[i], value[i]
    ax.scatter(x, y, s=200, c=MC[budget[nm]], alpha=.85, edgecolor="w", zorder=3)
    off = (6, 6) if nm!="自动巡检日报" else (6,-14)
    ax.annotate(nm, (x,y), textcoords="offset points", xytext=off, fontsize=9)

xs = np.linspace(0.1, 15, 100)
for r,ls in [(10,"--"),(3,":")]:
    ax.plot(xs, r*xs, ls=ls, c="gray", lw=1)
    ax.text(14.5, r*14.5, f"ROI={r}", fontsize=8, c="gray", rotation=18, ha="right")

ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("年化成本（万元/年，对数轴）"); ax.set_ylabel("年化价值（万元/年，对数轴）")
ax.set_title("实验4：预算象限图 —— 蓝色=费用预算行（当期答辩），橙色=资本预算行（二期答辩）")
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([],[],marker="o",ls="",c=MC["expense"],label="费用预算行（运维/安全科目）"),
                   Line2D([],[],marker="o",ls="",c=MC["capital"],label="资本预算行（资产运营科目）")],
          loc="upper left", fontsize=9)
fig.tight_layout(); fig.savefig("d5_budget_quadrant.png", dpi=110); plt.show()

print("回本周期速算：")
for i,nm in enumerate(names):
    print(f"  {nm:<10s} 年化成本 {cost[i]:5.2f} 万 | 首年价值 {value[i]*p_sys if budget[nm]=='expense' else value[i]:6.1f} 万 | 回本 {cost[i]/(value[i]) *12 if value[i]>0 else float('inf'):6.1f} 个月（名义）")